# Grad-CAM Visualization on Video Frames

Overlays Grad-CAM heatmaps on original video frames to visualize which body regions are
important for exercise classification. Uses the `action3d` package (`src/action3d/`).

**Requires:** the private dataset under `data/` and a trained checkpoint in `output/`
(run `training.ipynb` first, or place `action_3dcnn_model.pth` there manually).

In [ ]:
from collections import Counter
from pathlib import Path

import torch
from tqdm import tqdm

from action3d import GradCAM, load_checkpoint, load_label_file, load_skeleton_file, load_split_file
from action3d.visualization import visualize_gradcam_on_video

BASE_DIR = Path('..').resolve()
DATA_DIR = BASE_DIR / 'data'
SKELETON_DIR = DATA_DIR / 'dataset' / 'skeleton' / 'yolo_pose_csv'
VIDEO_DIR = DATA_DIR / 'dataset' / 'anon'
LABEL_DIR = DATA_DIR / 'label'
SPLIT_FILE = DATA_DIR / 'split.csv'
OUTPUT_DIR = BASE_DIR / 'output'
MODEL_PATH = OUTPUT_DIR / 'action_3dcnn_model.pth'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Model exists: {MODEL_PATH.exists()}")

## 1. Load Model

In [ ]:
model, checkpoint = load_checkpoint(MODEL_PATH, device)
CONFIG = checkpoint['config']
label_to_idx = checkpoint['label_to_idx']
idx_to_label = checkpoint['idx_to_label']

print(f"Loaded model with {checkpoint['num_classes']} classes")
print(f"Test accuracy: {checkpoint['test_acc']:.2f}%")

## 2. Find Test Samples with Matching Videos

In [ ]:
train_files, test_files = load_split_file(SPLIT_FILE)
MAX_SAMPLES_TO_VISUALIZE = 5
test_samples_with_video = []

for file_name in tqdm(sorted(test_files), desc='Finding samples'):
    if len(test_samples_with_video) >= MAX_SAMPLES_TO_VISUALIZE:
        break

    skeleton_path = SKELETON_DIR / f'{file_name}.csv'
    video_path = VIDEO_DIR / f'{file_name}.mp4'
    label_path = LABEL_DIR / f'{file_name}.csv'

    if not skeleton_path.exists() or not video_path.exists() or not label_path.exists():
        continue

    skeleton_data, _ = load_skeleton_file(skeleton_path)
    labels = load_label_file(label_path)

    min_len = min(len(skeleton_data), len(labels))
    skeleton_data = skeleton_data[:min_len]
    labels = labels[:min_len]

    for start_idx in range(0, len(skeleton_data) - CONFIG['sequence_length'] + 1, CONFIG['stride']):
        seq_labels = labels[start_idx:start_idx + CONFIG['sequence_length']]
        exercise_labels = seq_labels[seq_labels != -1]
        if len(exercise_labels) == 0:
            continue

        majority_label = Counter(exercise_labels).most_common(1)[0][0]
        if majority_label not in label_to_idx:
            continue

        test_samples_with_video.append({
            'file_name': file_name,
            'video_path': video_path,
            'skeleton_data': skeleton_data[start_idx:start_idx + CONFIG['sequence_length']],
            'start_frame': start_idx,
            'true_label': majority_label,
            'true_class': label_to_idx[majority_label],
        })
        break  # one sample per file

print(f"Found {len(test_samples_with_video)} test samples with matching videos")

## 3. Generate Video Grad-CAM Visualizations

`SKELETON_OFFSET_X` shifts the drawn skeleton horizontally (normalized coordinates) to align it with the person in the frame.

In [ ]:
gradcam = GradCAM(model, target_layer_idx=-1)
SKELETON_OFFSET_X = 0.2

for i, sample in enumerate(test_samples_with_video):
    print(f"\nSample {i + 1}/{len(test_samples_with_video)}: {sample['file_name']}")

    seq_data = sample['skeleton_data'].copy()  # (T, K, 3)
    input_tensor = torch.FloatTensor(seq_data.transpose(2, 0, 1)).unsqueeze(0).to(device)

    cam, pred_class, pred_prob = gradcam.generate_cam(input_tensor)
    print(f"  True: Exercise {sample['true_label']} | Predicted: Exercise {idx_to_label[pred_class]} ({pred_prob:.1%})")

    save_path = OUTPUT_DIR / f"gradcam_video_{sample['file_name']}.png"
    visualize_gradcam_on_video(
        video_path=sample['video_path'],
        skeleton_data=seq_data,
        cam=cam,
        start_frame=sample['start_frame'],
        pred_class=pred_class,
        true_class=sample['true_class'],
        pred_prob=pred_prob,
        idx_to_label=idx_to_label,
        num_frames_to_show=8,
        save_path=save_path,
        offset_x=SKELETON_OFFSET_X,
    )

gradcam.remove_hooks()
print(f"\nAll video Grad-CAM visualizations saved to: {OUTPUT_DIR}")